# Frozen final experiment — Google Colab

Use a **T4 GPU** runtime. This notebook runs the centralized repository implementation only. It requires a deliberate opt-in before the explicit `--full` command can execute.

In [ ]:
REPO_URL = "https://github.com/<OWNER>/<REPOSITORY>.git"
REPO_DIR = "llm-prompt-sensitivity"
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git rev-parse HEAD
!python -m pip install -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime, then reconnect."
print(torch.cuda.get_device_name(0))
!python -m pytest -q

In [ ]:
# Deliberate safety gate. Review the frozen methodology and set to True only to authorize the final run.
RUN_FINAL = False
assert RUN_FINAL is True, "Final run is blocked. Set RUN_FINAL = True only after explicit approval."

# Ensure this shell runs from the cloned repository, not Colab's default directory.
%cd /content/$REPO_DIR

# Explicit final-run invocation: 300 seeded questions × 5 variants = 1,500 generations.
!PYTHONPATH=. python -m src.runner --full

In [ ]:
from pathlib import Path
assert len(Path("outputs/final/generations.jsonl").read_text().splitlines()) == 1500
!PYTHONPATH=. python scripts/analyze.py outputs/final/generations.jsonl
!PYTHONPATH=. python scripts/plot_results.py outputs/final/generations.jsonl

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("outputs/final_artifacts", "zip", root_dir="outputs", base_dir="final")
files.download(archive)